# 4. Score predictions with DockQ

Compare each predicted complex returned by notebook 2 against an experimental ground-truth structure, and report a DockQ score per prediction.

**Ground-truth structures.** Drop your experimental mmCIF files into `examples/exp_structures/` (or set `GROUND_TRUTH_DIR` to point elsewhere), one file per row in your input CSV. For DockQ scoring, each filename must match that row's `exp_structure` value. This repo ships four example ground-truth structures (matching `examples/structure_determination_input.csv`) so the default configuration works out of the box; replace those with your own files when running on your data. Predictions with no matching ground-truth file are skipped with a warning.

**Only the antibody-antigen interface(s) are scored.** The intra-antibody (VH-VL) interface is excluded because it packs the same way regardless of whether the antibody binds the antigen, so including it would inflate the headline number without telling you anything about binding quality.

**Antigen-chain support.** This notebook currently resolves one antigen chain per prediction. Use single-chain antigen inputs for this bundled DockQ workflow.

One row is emitted per prediction CIF (no aggregation across samples). DockQ scoring uses the public [`DockQ`](https://github.com/bjornwallner/DockQ) PyPI package.

## Inputs

- `PREDICTIONS_DIR` — the `cifs/` subdir of a notebook-2 run.
- `GROUND_TRUTH_DIR` — folder of experimental `<name>.cif` files. Defaults to `./examples/exp_structures/` (the bundled examples); set to your own folder for production use.
- `CONFIDENCE_DIR` — optional. The `confidence/` subdir from the same notebook-2 run; if present, `iptm` and `ranking_score` are included alongside each DockQ row.
- `INPUT_CSV` — the input CSV that drove inference, used to resolve which chain is the antigen in each prediction. The bundled workflow expects one antigen chain per row.

## Outputs

- `dockq_per_prediction.csv` — one row per prediction CIF: `input_name, prediction_id, dockq, num_interfaces, iptm, ranking_score, pred_path, native_path`.
- `dockq_details/` — full per-interface JSON from DockQ for each scored pair.

In [ ]:
import json
import re
import sys
import traceback
from pathlib import Path

import pandas as pd

sys.path.append("src")
from az_adapter import csv_to_jobs
from dockq_scoring import dockq
from epitope import find_antigen_chain_id

# Inputs.
INPUT_CSV = Path("examples/structure_determination_input.csv")
GROUND_TRUTH_DIR = Path("./examples/exp_structures")

# By default, score the most-recent invoke_* run under outputs/. Override
# PREDICTIONS_DIR / CONFIDENCE_DIR with absolute paths if you want a specific run.
_invoke_runs = sorted(Path("./outputs").glob("invoke_*"))
PREDICTIONS_DIR = _invoke_runs[-1] / "cifs" if _invoke_runs else Path("./outputs/invoke_<timestamp>/cifs")
CONFIDENCE_DIR = _invoke_runs[-1] / "confidence" if _invoke_runs else Path("./outputs/invoke_<timestamp>/confidence")

# Outputs.
OUTPUT_CSV = Path("./outputs/dockq_per_prediction.csv")
DETAIL_DIR = Path("./outputs/dockq_details")

# How many residue-level mismatches between model and native chains we tolerate
# when building the chain mapping. 4 matches the FoldBench bench_hhsearch setting.
ALLOWED_MISMATCHES = 4


In [ ]:
# Antigen sequence per input_name, from the CSV that drove inference. We use
# this to resolve which chain in each predicted CIF is the antigen (so DockQ
# can filter to Ab-Ag interfaces only).
name_to_antigen = {
    job["name"]: job["sequences"][0]["proteinChain"]["sequence"]
    for job in csv_to_jobs(INPUT_CSV)
}

# Strip the "batch_NNN_nM_aaK_" prefix that notebook 2 prepends to each CIF.
BATCH_PREFIX_RE = re.compile(r"^batch_\d+_n\d+_aa\d+_")


def input_name_from_cif(cif_path: Path) -> str:
    return BATCH_PREFIX_RE.sub("", cif_path.stem)


def confidence_for(cif_path: Path) -> dict:
    if not CONFIDENCE_DIR.exists():
        return {}
    conf_path = CONFIDENCE_DIR / f"{cif_path.stem}.json"
    if not conf_path.exists():
        return {}
    return json.loads(conf_path.read_text()).get("summary", {}) or {}


prediction_paths = sorted(PREDICTIONS_DIR.glob("*.cif"))
print(f"Found {len(prediction_paths)} prediction CIFs in {PREDICTIONS_DIR}")
if any(input_name_from_cif(p) not in name_to_antigen for p in prediction_paths): raise ValueError("PREDICTIONS_DIR contains CIFs not present in INPUT_CSV; set both paths to the same run.")

pairs = []
missing = []
for cif in prediction_paths:
    input_name = input_name_from_cif(cif)
    native = GROUND_TRUTH_DIR / f"{input_name}.cif"
    if native.exists():
        pairs.append((input_name, cif, native))
    else:
        missing.append((cif.name, str(native)))

print(f"Matched to ground truth: {len(pairs)}")
if missing:
    print(f"Skipping {len(missing)} predictions with no ground-truth match (first few):")
    for cif_name, native_path in missing[:5]:
        print(f"  {cif_name} -> expected {native_path}")


In [ ]:
DETAIL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

rows = []
for input_name, pred_path, native_path in pairs:
    conf = confidence_for(pred_path)
    iptm = conf.get("iptm")
    ranking_score = conf.get("ranking_score")
    try:
        antigen_chain_id = find_antigen_chain_id(str(pred_path), name_to_antigen[input_name])
        info = dockq(
            model_path=str(pred_path),
            native_path=str(native_path),
            allowed_mismatches=ALLOWED_MISMATCHES,
            antigen_chain_id=antigen_chain_id,
        )
    except Exception:
        print(f"DockQ failed for {pred_path.name}:")
        traceback.print_exc()
        rows.append({
            "input_name": input_name,
            "prediction_id": pred_path.stem,
            "dockq": None,
            "num_interfaces": None,
            "iptm": iptm,
            "ranking_score": ranking_score,
            "pred_path": str(pred_path),
            "native_path": str(native_path),
        })
        continue

    detail = DETAIL_DIR / f"{pred_path.stem}.json"
    detail.write_text(
        json.dumps(
            {
                "model": info["model"],
                "native": info["native"],
                "best_dockq": info["best_dockq"],
                "best_result": info["best_result"],
            },
            indent=2,
            default=str,
        )
    )

    interfaces = list(info["best_result"].keys()) if info["best_result"] else []
    rows.append({
        "input_name": input_name,
        "prediction_id": pred_path.stem,
        "dockq": info["best_dockq"],
        "num_interfaces": len(interfaces),
        "iptm": iptm,
        "ranking_score": ranking_score,
        "pred_path": str(pred_path),
        "native_path": str(native_path),
    })
    print(f"{input_name}: dockq={info['best_dockq']:.3f} ({len(interfaces)} interfaces)")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nWrote {len(df)} rows to {OUTPUT_CSV}")
df.head()


## Interpretation

- **`dockq`** — mean per-interface DockQ across the Ab-Ag interfaces in the best chain mapping. Bounded [0, 1]; higher is better. For a VH/VL antibody this is the mean of (heavy-antigen, light-antigen) DockQ; for a single-chain antibody it's just the one Ab-Ag interface value.
- **`num_interfaces`** — count of Ab-Ag interfaces retained after filtering. Typically 1 for single-chain antibodies, 2 for VH/VL.

For per-interface detail (LRMSD, iRMSD, per-interface DockQ), look at `DETAIL_DIR/<prediction_id>.json`.